In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# 라이브러리 설치/불러오기
!pip install folium -q

import pandas as pd
import folium

# 위험도 결과 CSV 불러오기

risk_path = '/content/drive/MyDrive/서울시 빅데이터 공모전/data/processed/subway_risk_result.csv'

risk_df = pd.read_csv(risk_path, encoding='utf-8-sig')

print(risk_df.shape)
print(risk_df.columns.tolist())
risk_df.head()

(1200, 14)
['역명_clean', '호선명', 'time_group', 'Risk_Score', 'Risk', '사고수', '사고발생', '위험유형', '주요위험원인', '혼잡기여도', '유입기여도', '구조기여도', '환승기여도', '추천대응전략']


,역명_clean,호선명,time_group,Risk_Score,Risk,사고수,사고발생,위험유형,주요위험원인,혼잡기여도,유입기여도,구조기여도,환승기여도,추천대응전략
0,사당,2호선,퇴근시간,89.612410,0.896124,11,1,환승형,환승형,17.577361,16.471606,24.676271,41.274762,"환승 동선 분리, 환승 통로 안전요원 배치, 환승 안내 표지 강화"
1,사당,2호선,낮시간,89.576990,0.895770,12,1,환승형,환승형,17.273867,15.784539,25.046900,41.894694,"환승 동선 분리, 환승 통로 안전요원 배치, 환승 안내 표지 강화"
2,사당,4호선,낮시간,89.482670,0.894827,8,1,환승형,환승형,11.508750,25.662743,19.450347,43.378160,"환승 동선 분리, 환승 통로 안전요원 배치, 환승 안내 표지 강화"
3,종로3가,1호선,낮시간,89.417564,0.894176,4,1,구조형,구조형,12.316445,19.858253,33.977756,33.847546,"승강장 유도선 정비, 안전표지 보강, 시설 구조 점검"
4,신도림,2호선,낮시간,89.229870,0.892299,10,1,환승형,환승형,17.240165,11.438121,15.067968,56.253747,"환승 동선 분리, 환승 통로 안전요원 배치, 환승 안내 표지 강화"


In [ ]:
# 역 좌표 CSV 불러오기

coord_path = '/content/drive/MyDrive/서울시 빅데이터 공모전/data/raw/지하철역좌표.csv'

coord_df = pd.read_csv(coord_path, encoding='cp949')

print(coord_df.shape)
print(coord_df.columns.tolist())
coord_df.head()

(783, 5)
['역사_ID', '역사명', '호선', '위도', '경도']


,역사_ID,역사명,호선,위도,경도
0,9010,동탄,수도권 광역급행철도,37.20034,127.09569
1,9009,구성,수도권 광역급행철도,37.29913,127.10389
2,9008,성남,수도권 광역급행철도,37.39467,127.12058
3,9007,수서,수도권 광역급행철도,37.48637,127.10161
4,9006,삼성,수도권 광역급행철도,37.50887,127.06324


In [ ]:
# 컬럼명 통일
# 좌표 파일의 '역사명' 컬럼을
# 위험도 데이터의 '역명_clean'과 맞추기 위해 변경
coord_df = coord_df.rename(columns={
    '역사명': '역명_clean'
})

In [ ]:
# 역명 정리 함수
# 괄호, 공백 등 제거해서
# 서로 다른 표기 통일
def clean_station_name(name):
    name = str(name).strip()
    name = name.replace(" ", "")
    name = name.split("(")[0]
    return name

In [ ]:
# merge용 key 생성
risk_df['역명_key'] = risk_df['역명_clean'].apply(clean_station_name)
coord_df['역명_key'] = coord_df['역명_clean'].apply(clean_station_name)

In [ ]:
# 좌표 중복 제거
# 같은 역이 여러 호선에 존재하므로
# 역명 기준 평균 좌표 사용
coord_one = (
    coord_df
    .dropna(subset=['위도', '경도'])
    .groupby('역명_key', as_index=False)
    .agg({
        '위도': 'mean',
        '경도': 'mean'
    })
)

print(coord_one.shape)
coord_one.head()

(654, 3)


,역명_key,위도,경도
0,4.19민주묘지,37.649502,127.013684
1,가능,37.748577,127.044213
2,가락시장,37.492566,127.118077
3,가산디지털단지,37.480959,126.882619
4,가양,37.561391,126.854456


In [ ]:
# 위험도 데이터 + 좌표 병합
map_df = risk_df.merge(
    coord_one,
    on='역명_key',
    how='left'
)

print("병합 결과:", map_df.shape)
print("좌표 없는 행 개수:", map_df[['위도', '경도']].isna().any(axis=1).sum())

map_df.head()

병합 결과: (1200, 17)
좌표 없는 행 개수: 8


,역명_clean,호선명,time_group,Risk_Score,Risk,사고수,사고발생,위험유형,주요위험원인,혼잡기여도,유입기여도,구조기여도,환승기여도,추천대응전략,역명_key,위도,경도
0,사당,2호선,퇴근시간,89.612410,0.896124,11,1,환승형,환승형,17.577361,16.471606,24.676271,41.274762,"환승 동선 분리, 환승 통로 안전요원 배치, 환승 안내 표지 강화",사당,37.476746,126.981597
1,사당,2호선,낮시간,89.576990,0.895770,12,1,환승형,환승형,17.273867,15.784539,25.046900,41.894694,"환승 동선 분리, 환승 통로 안전요원 배치, 환승 안내 표지 강화",사당,37.476746,126.981597
2,사당,4호선,낮시간,89.482670,0.894827,8,1,환승형,환승형,11.508750,25.662743,19.450347,43.378160,"환승 동선 분리, 환승 통로 안전요원 배치, 환승 안내 표지 강화",사당,37.476746,126.981597
3,종로3가,1호선,낮시간,89.417564,0.894176,4,1,구조형,구조형,12.316445,19.858253,33.977756,33.847546,"승강장 유도선 정비, 안전표지 보강, 시설 구조 점검",종로3가,37.571517,126.991314
4,신도림,2호선,낮시간,89.229870,0.892299,10,1,환승형,환승형,17.240165,11.438121,15.067968,56.253747,"환승 동선 분리, 환승 통로 안전요원 배치, 환승 안내 표지 강화",신도림,37.508874,126.891114


In [ ]:
# 좌표 없는 역 확인
map_df[map_df[['위도', '경도']].isna().any(axis=1)][
    ['역명_clean', '호선명']
].drop_duplicates()

,역명_clean,호선명
625,당고개,4호선
660,자양,7호선


In [ ]:
# Floium 지도 생성

!pip install folium -q

import folium

# 좌표 있는 데이터만 사용
map_data = map_df.dropna(subset=['위도', '경도']).copy()

# 역별 최고 Risk Score 기준으로 대표값 선택
station_map_data = (
    map_data
    .sort_values('Risk_Score', ascending=False)
    .groupby('역명_clean', as_index=False)
    .first()
)

# Risk Score별 색상 함수
def get_color(score):
    if score >= 85:
        return 'red'
    elif score >= 75:
        return 'orange'
    elif score >= 60:
        return 'yellow'
    else:
        return 'green'

# 서울 중심 지도 생성
m = folium.Map(
    location=[37.5665, 126.9780],
    zoom_start=11
)

# 지도에 역별 위험도 표시
for _, row in station_map_data.iterrows():
    folium.CircleMarker(
        location=[row['위도'], row['경도']],
        radius=6 + row['Risk_Score'] / 20,
        color=get_color(row['Risk_Score']),
        fill=True,
        fill_color=get_color(row['Risk_Score']),
        fill_opacity=0.7,
        popup=f"""
        <b>{row['역명_clean']}</b><br>
        호선: {row['호선명']}<br>
        시간대: {row['time_group']}<br>
        Risk Score: {row['Risk_Score']:.2f}<br>
        위험유형: {row['위험유형']}<br>
        추천전략: {row['추천대응전략']}
        """
    ).add_to(m)

m